# 04｜相对位置偏置 Relative Position Bias

前两课已经拆解了 Window Attention 和 Shifted Window。现在还有一个问题：窗口内的 Self-Attention 虽然能比较 token 内容，却怎样知道两个 tokens 在空间上是上下相邻、左右相邻，还是相隔很远？

Swin Transformer 使用相对位置偏置，把窗口内两个 tokens 的相对方向和距离加入注意力分数。

这一课只讲相对位置偏置的原理、参数表、索引矩阵和 shape，不进入源码实现。

最重要的一句话：

> **相对位置偏置：根据两个 Patch 的相对方向和距离，给 Attention 分数额外加一个可学习的值。**

## 1. 为什么 QK 相似度还不够

Self-Attention 首先通过 Q 与 K 的点积衡量两个 tokens 的内容关系：

$$
S_{ij}=\frac{q_i k_j^{\top}}{\sqrt{d_{\mathrm{head}}}}
$$

这个分数主要回答：token $i$ 的查询与 token $j$ 的特征是否匹配。

但是，仅靠内容相似度不能明确区分下面这些空间关系：

- token $j$ 在 token $i$ 的正上方；
- token $j$ 在 token $i$ 的右下方；
- 两个 tokens 相邻；
- 两个 tokens 位于窗口的两端。

图像中的方向和距离通常很重要，因此 Swin 需要把相对空间关系额外加入 Attention。

## 2. 什么是相对位置

设查询 token 的二维坐标为 $(h_q,w_q)$，被查询 token 的坐标为 $(h_k,w_k)$。定义它们的相对位移为：

$$
\Delta h=h_q-h_k,\qquad\Delta w=w_q-w_k
$$

相对位置不关心两个 tokens 在整张图片中的绝对坐标，只关心它们相差多少。

例如：

| 相对位移 | 含义 |
|---|---|
| $(0,0)$ | 查询自己 |
| $(0,-1)$ | 按当前定义，被查询 token 位于查询 token 右侧一格 |
| $(1,0)$ | 被查询 token 位于查询 token 上方一格 |
| $(2,-3)$ | 两者在高度和宽度方向分别相差 2 与 3 格 |

如果交换查询和被查询 token，位移符号也会反转。因此 $(0,-1)$ 与 $(0,1)$ 是两种不同关系。

## 3. 为什么 $7\times7$ 窗口有 169 种相对位移

窗口边长为 $M=7$。任意两个 tokens 在高度方向上的最大距离是 6，因此 $\Delta h$ 的取值范围是：

$$
\Delta h\in\{-6,-5,\ldots,0,\ldots,5,6\}
$$

一共有：

$$
2M-1=2\times7-1=13\text{ 种高度位移}
$$

宽度方向同样有 13 种位移，所以二维相对位移的组合数量为：

$$
(2M-1)^2=13\times13=169
$$

这里的 169 不是 token 对的数量，而是所有 token 对可能具有的相对位移类型数量。

![相对位置偏置的查表过程](images/05_relative_position_bias.svg)

## 4. 为什么不是为 2401 个 token 对分别学习参数

一个 $7\times7$ 窗口有 49 个 tokens，因此有：

$$
49\times49=2401\text{ 个有方向的 token 对}
$$

但是许多 token 对拥有相同的相对位移。

例如，窗口中每一对水平相邻且方向相同的 tokens，都可以共享同一种相对位置偏置。模型不需要为每一对绝对坐标分别学习参数。

因此 Swin 只为 169 种相对位移分别保存偏置，再让 2401 个 token 对按照自己的相对位移查找对应值。

这种共享方式带来两点好处：

1. 参数量更少。
2. 相同方向和距离的空间关系采用一致的先验。

## 5. 相对位置偏置表的 shape

不同注意力头可以学习不同的空间偏好，因此每一种相对位移都要为每个 head 保存一个偏置值。

设注意力头数为 $h$，偏置表形状为：

$$
(2M-1)^2\times h
$$

Swin-T Stage 1 使用 $M=7$、$h=3$，因此：

$$
B_{\mathrm{table}}\in\mathbb{R}^{169\times3}
$$

总共有：

$$
169\times3=507\text{ 个可学习偏置参数}
$$

同一个 Attention 模块中的所有窗口共享这张表，但不同 head 读取不同列。

## 6. 为什么还需要相对位置索引矩阵

偏置表只有 169 行，但窗口注意力分数矩阵有 $49\times49$ 个位置。模型需要知道每个 token 对应该查询偏置表中的哪一行。

因此会预先建立一个相对位置索引矩阵：

$$
I_{\mathrm{rel}}\in\mathbb{N}^{49\times49}
$$

矩阵中的元素 $I_{\mathrm{rel}}[i,j]$ 表示：第 $i$ 个查询 token 与第 $j$ 个被查询 token 的相对位移，对应偏置表中的哪一行。

这个索引矩阵不是可学习参数。窗口大小固定后，它只由坐标关系决定，可以提前计算并重复使用。

## 7. 二维位移怎样映射成一维索引

原始位移 $\Delta h$ 和 $\Delta w$ 都可能为负数，不能直接作为表格索引。第一步先各自加上 $M-1$：

$$
\Delta h'=\Delta h+(M-1),\qquad
\Delta w'=\Delta w+(M-1)
$$

对于 $M=7$，两个坐标都会从 $[-6,6]$ 平移到 $[0,12]$。

接着把 $13\times13$ 的二维位置展平成一维索引：

$$
r=\Delta h'\,(2M-1)+\Delta w'
$$

因此：

$$
r\in\{0,1,\ldots,168\}
$$

不同实现可能采用相反的位移符号约定，但“二维相对位移映射到 169 个表项”的原理相同。

## 8. 从偏置表得到每个 head 的偏置矩阵

使用 $49\times49$ 的索引矩阵查询偏置表后，每一对 tokens 都取得对应的各 head 偏置。

查询后的形状先是：

$$
49\times49\times h
$$

为了与多头注意力分数对齐，再把 head 维度移到前面：

$$
B_{\mathrm{rel}}\in\mathbb{R}^{h\times49\times49}
$$

Stage 1 中就是：

$$
B_{\mathrm{rel}}\in\mathbb{R}^{3\times49\times49}
$$

三个 head 可以对同一种相对位移学习三个不同偏置，因此它们可能关注不同空间关系。

## 9. Q、K、V 与相对位置偏置 $B$ 怎样对齐

这里的 $B$ 专门表示相对位置偏置。为了避免把它和 batch 混淆，下面用 $G$ 表示一次参与计算的窗口数量。

以 Stage 1 为例：窗口大小为 $7\times7$，所以每个窗口有 $N=49$ 个 tokens；通道数为 96；注意力头数为 3；每个 head 的特征维度为 $d_{\mathrm{head}}=96\div3=32$。拆成多头后：

$$
Q,K,V\in\mathbb{R}^{G\times3\times49\times32}
$$

对 $K$ 的最后两个维度转置，再与 $Q$ 相乘：

$$
\begin{aligned}
Q&:G\times3\times49\times32 \\
K^{\top}&:G\times3\times32\times49 \\
QK^{\top}&:G\times3\times49\times49
\end{aligned}
$$

这里每个 head 的 $49\times49$ 分数矩阵表示 49 个 Query tokens 分别与 49 个 Key tokens 的内容相似度。元素 $(i,j)$ 对应“token $i$ 关注 token $j$”这一对有方向的关系。

### 9.1 不能把 $13\times13$ 直接加到 $49\times49$ 上

$13\times13$ 表示 169 种可能的相对位移，不是 49 个 tokens 两两之间的完整偏置矩阵。因此它不能直接与 $QK^{\top}$ 的 $49\times49$ 相加。

模型要先为 2401 个有方向的 token 对逐一查询偏置表：

$$
\begin{aligned}
B_{\mathrm{table}}&:169\times3 \\
I_{\mathrm{rel}}&:49\times49 \\
\text{查表后}&:49\times49\times3 \\
\text{调整维度后 }B_{\mathrm{rel}}&:3\times49\times49
\end{aligned}
$$

例如，token $i$ 和 token $j$ 的相对位移为 $(3,1)$，就查询偏置表中代表 $(3,1)$ 的那一行。若某个 head 查到的偏置是 $0.2$，而这对 tokens 的 QK 内容分数是 $0.7$，相加后就是 $0.9$。

### 9.2 最终按 head、Query、Key 三个维度逐项相加

给相对位置偏置补上一个大小为 1 的窗口维度后：

$$
\underbrace{G\times3\times49\times49}_{QK^{\top}}
+
\underbrace{1\times3\times49\times49}_{B_{\mathrm{rel}}}
\longrightarrow
G\times3\times49\times49
$$

两个张量的 head、Query token、Key token 三个维度完全相同。最前面的 1 会广播到全部 $G$ 个窗口，因为同一个 Attention 模块中的所有窗口共享相同的相对位置偏置表。

对具体的一对 tokens，执行的是：

$$
S[g,a,i,j]=\frac{Q[g,a,i,:]\cdot K[g,a,j,:]}{\sqrt{32}}+B_{\mathrm{rel}}[a,i,j]
$$

其中 $g$ 是窗口编号，$a$ 是 head 编号，$i$ 是 Query token，$j$ 是 Key token。QK 分数回答“内容是否匹配”，相对位置偏置回答“根据方向和距离，要额外鼓励还是抑制这次关注”。

经过 Softmax 得到注意力权重后，再与 $V$ 相乘：

$$
(G\times3\times49\times49)\cdot(G\times3\times49\times32)
\longrightarrow G\times3\times49\times32
$$

因此，$B$ 不与 $Q$、$K$ 或 $V$ 单独相加；它只与 $QK^{\top}$ 产生的 token 两两分数矩阵对齐并相加。

## 10. 偏置值为正或为负意味着什么

相对位置偏置直接加在 Softmax 之前的 logits 上。

- 正偏置会提高对应 token 对的 logit，使其更容易获得较大注意力权重。
- 负偏置会降低对应 logit，使其更不容易被关注。
- 偏置为 0 表示不额外提高或压低该空间关系。

这些偏置不是人工规定的。它们与 QKV 投影等参数一样，通过训练和反向传播自动学习。

例如，某个 head 可能逐渐偏好邻近位置，另一个 head 可能更关注斜对角方向或较远位置。这里只能说模型具备这种学习能力，不能在训练前预先断定每个 head 一定学到什么。

## 11. 相对位置偏置与 Attention Mask 的区别

相对位置偏置和 Shifted Window 的 mask 都会加到 Attention logits 上，但作用完全不同。

| 对比 | 相对位置偏置 | Attention Mask |
|---|---|---|
| 是否学习 | 是 | 否 |
| 主要作用 | 表达方向和距离偏好 | 禁止虚假的循环边界通信 |
| 典型数值 | 训练得到的正数或负数 | 0 或很大的负数 |
| W-MSA 是否需要 | 需要 | 通常不需要 |
| SW-MSA 是否需要 | 需要 | 需要 |

在 SW-MSA 中，完整形式可以写为：

$$
A=\operatorname{softmax}\left(
\frac{QK^{\top}}{\sqrt{d_{\mathrm{head}}}}
+B_{\mathrm{rel}}+\mathcal{M}
\right)
$$

$B_{\mathrm{rel}}$ 调整空间偏好，$\mathcal{M}$ 强制屏蔽不允许的连接。

## 12. 相对位置偏置与 ViT 位置编码的区别

ViT 常在进入 Encoder 前，给每个 token 加上一个绝对位置向量。它回答的是“这个 token 位于第几个位置”。

Swin 的相对位置偏置直接加入注意力 logits，回答的是“查询 token 与被查询 token 在空间上相差多少”。

| 对比 | ViT 常见绝对位置编码 | Swin 相对位置偏置 |
|---|---|---|
| 描述对象 | 单个 token 的绝对位置 | 两个 tokens 的相对位置 |
| 加入位置 | token 表示 | Attention logits |
| 是否区分方向和距离 | 间接学习 | 直接按相对位移查表 |
| 是否在窗口间共享 | 通常对应完整序列位置 | 同一模块的所有窗口共享 |

二者都用于补充位置信息，但注入方式和表达重点不同。

## 13. Swin-T 不同 Stage 的偏置表

Swin-T 各 Stage 通常都使用 $7\times7$ 窗口，因此相对位移类型始终是 169 种；但注意力头数逐 Stage 增加。

| Stage | head 数量 | 单个 Attention 模块的偏置表 shape | 参数量 |
|---|---:|---:|---:|
| Stage 1 | 3 | $169\times3$ | 507 |
| Stage 2 | 6 | $169\times6$ | 1014 |
| Stage 3 | 12 | $169\times12$ | 2028 |
| Stage 4 | 24 | $169\times24$ | 4056 |

这里说的是单个 Attention 模块。通常每个 Swin Block 拥有自己的可学习偏置表，而该模块内部的所有窗口共享它。

偏置参数量相对于整个模型很小，但它为 Attention 提供了明确的二维相对空间信息。

## 14. 完整 shape 路线

以 Stage 1 的单个 Attention 模块为例：

$$
\begin{aligned}
\text{相对位移类型数量}\quad&13\times13=169 \\n\text{偏置表}\quad&169\times3 \\n\text{相对位置索引}\quad&49\times49 \\n\text{查表结果}\quad&49\times49\times3 \\n\text{调整维度顺序}\quad&3\times49\times49 \\n\text{补窗口维度}\quad&1\times3\times49\times49 \\
\text{广播到所有窗口}\quad&G\times3\times49\times49
\end{aligned}
$$

最后一行与 $QK^{\top}$ 注意力分数形状一致，可以逐元素相加。这里用 $G$ 表示窗口数量，$B$ 只表示相对位置偏置。

注意不要混淆：169 是相对位移类型数量，2401 是窗口内有方向的 token 对数量，$49\times49$ 是索引矩阵与单个 head 偏置矩阵的形状。

## 15. 常见误区

### 误区一：相对位置偏置就是注意力权重

偏置只是在 Softmax 前调整 logits。最终注意力权重还同时受到 QK 内容相似度影响。

### 误区二：每个 token 对都有独立偏置参数

2401 个 token 对通过相对位置索引共享 169 种位移参数。

### 误区三：所有 head 使用同一个偏置值

同一种位移会为不同 head 分别保存偏置，每个 head 可以学习不同空间偏好。

### 误区四：相对位置索引也要训练

索引只由坐标关系决定，不参与学习；真正学习的是偏置表中的数值。

## 16. 本节小结

这一课需要掌握下面七点：

1. QK 点积主要描述内容关系，相对位置偏置补充方向和距离信息。
2. $7\times7$ 窗口有 49 个 tokens、2401 个有方向的 token 对，但只有 169 种相对位移。
3. Stage 1 的偏置表 shape 为 $169\times3$。
4. $49\times49$ 的相对位置索引矩阵负责为每个 token 对查找表项。
5. 查表结果调整为 $3\times49\times49$，再广播到所有窗口。
6. $13\times13$ 偏置表不能直接加到 $49\times49$ 的 QK 分数上；必须先逐对查表，形成每个 head 的 $49\times49$ 偏置矩阵。
7. 相对位置偏置是可学习的空间偏好，Attention Mask 是不可学习的强制屏蔽规则。

下一课适合学习 Patch Merging 的具体拼接过程、shape 变化和它与 CNN 下采样的关系。

## 17. 自测问题

1. QK 内容相似度为什么不能完整表达二维空间关系？
2. 相对位置 $(0,-1)$ 与 $(0,1)$ 为什么不同？
3. $7\times7$ 窗口的单个方向为什么有 13 种相对位移？
4. 169、49 和 2401 分别表示什么？
5. 为什么具有相同相对位移的 token 对可以共享偏置？
6. Stage 1 的相对位置偏置表为什么是 $169\times3$？
7. 相对位置索引矩阵为什么是 $49\times49$？
8. 相对位置索引是否通过训练学习？
9. 查表结果为什么要调整为 $h\times49\times49$？
10. 相对位置偏置怎样广播到所有窗口？
11. 正偏置和负偏置分别怎样影响 Softmax 前的 logit？
12. 相对位置偏置与 Attention Mask 的作用有什么不同？
13. 为什么不能把 $13\times13$ 的偏置表直接加到 $49\times49$ 的 QK 分数矩阵上？
14. 相对位置偏置与 ViT 的绝对位置编码有什么不同？

### 自测参考答案

1. QK 主要比较 token 内容，不能直接指出两个 tokens 的相对方向和距离。
2. 它们方向相反，交换查询和被查询 token 会改变位移符号。
3. 最大位移为 6，取值从 -6 到 6，共 13 种。
4. 169 是相对位移类型数量，49 是窗口 token 数量，2401 是有方向的 token 对数量。
5. 相同方向和距离代表同一种空间关系，可以使用相同参数。
6. 共有 169 种位移，3 个 head 为每种位移分别学习偏置。
7. 49 个查询 tokens 都要为 49 个被查询 tokens 查找相对位置。
8. 不学习，它由窗口坐标关系预先确定。
9. 为了与多头注意力分数的 head、query、key 三个维度对齐。
10. 在最前面补一个大小为 1 的窗口维度，再沿窗口数量 $G$ 广播。
11. 正偏置提高 logit，负偏置降低 logit，从而影响 Softmax 权重。
12. 偏置学习空间偏好；mask 强制屏蔽循环移位产生的非法连接。
13. 因为 $13\times13$ 只表示 169 种相对位移类型；必须先让 2401 个 token 对按各自位移查表，得到 $49\times49$ 的偏置矩阵，形状才能与 QK 分数对齐。
14. 绝对位置编码加入 token 表示；相对位置偏置按 token 对的相对位移加入 Attention logits。